# S03 Seoul bike sharing: a first look (explained version)
Same notebook as `S03_understanding.ipynb`, but **every line of code is explained** in the text cell above it.
Read the explanation, run the cell (Shift + Enter), then compare the output with what the text said.

## 1. Load seoul_reference.csv: 8,760 rows, one per hour, Dec 2017–Nov 2018

`import pandas as pd` loads the **pandas** library, the standard Python tool for tables. `as pd` gives it a
short nickname, so later we write `pd.read_csv` instead of `pandas.read_csv`.

In [1]:
import pandas as pd

`pd.read_csv("../data/seoul_reference.csv")` reads the CSV file into a **DataFrame** (a table with named
columns). `../data/` means: go one folder up from this notebook, then into `data`. We store the table in a
variable called `df` (short for *data frame*).

In [2]:
df = pd.read_csv("../data/seoul_reference.csv")

`len(df)` returns the **number of rows**. We expect 8,760: one row per hour for a full year (365 × 24).

In [3]:
len(df)

8760

## 2. shape, head, dtypes: what does one row mean?

`df.shape` gives **(rows, columns)** as a pair. No brackets after `shape`: it's a property of the table, not
a function you call.

In [4]:
df.shape

(8760, 14)

`df.head()` shows the **first 5 rows**. This is the quickest way to see what the values look like.
(`df.head(10)` would show 10 rows, `df.tail()` the last 5.)

In [5]:
df.head()

,timestamp,rented_bike_count,temp_c,humidity_pct,wind_speed_m_s,visibility_m,dew_point_c,solar_radiation_mj_m2,rainfall_mm,snowfall_cm,season,holiday,functioning_day,split
0,2017-12-01 00:00,254,-5.2,37,2.2,20000,-17.6,0.0,0.0,0.0,Winter,No Holiday,Yes,train
1,2017-12-01 01:00,204,-5.5,38,0.8,20000,-17.6,0.0,0.0,0.0,Winter,No Holiday,Yes,train
2,2017-12-01 02:00,173,-6.0,39,1.0,20000,-17.7,0.0,0.0,0.0,Winter,No Holiday,Yes,train
3,2017-12-01 03:00,107,-6.2,40,0.9,20000,-17.6,0.0,0.0,0.0,Winter,No Holiday,Yes,train
4,2017-12-01 04:00,78,-6.0,36,2.3,20000,-18.6,0.0,0.0,0.0,Winter,No Holiday,Yes,train


`df.dtypes` lists the **data type of every column**: `int64` = whole numbers, `float64` = numbers with
decimals, `object` or `str` = text. A column that should be numeric but shows up as text is a warning sign.

In [6]:
df.dtypes

timestamp                    str
rented_bike_count          int64
temp_c                   float64
humidity_pct               int64
wind_speed_m_s           float64
visibility_m               int64
dew_point_c              float64
solar_radiation_mj_m2    float64
rainfall_mm              float64
snowfall_cm              float64
season                       str
holiday                      str
functioning_day              str
split                        str
dtype: object

**What one row means:** one hour in Seoul: how many bikes were rented in that hour, plus the weather at that
time. `timestamp`, `season`, `holiday`, `functioning_day` and `split` are text; everything else is numeric.

## 3. describe: which ranges look plausible?

`df.describe()` calculates **summary statistics** for every numeric column: `count` (values that are not
missing), `mean`, `std` (spread), `min`, the quartiles `25%` / `50%` (median) / `75%`, and `max`.
`.round(1)` rounds everything to one decimal so it is easier to read.
`.T` **transposes** the table (rows ↔ columns), so each column of the data becomes one readable row.

In [7]:
df.describe().round(1).T

,count,mean,std,min,25%,50%,75%,max
rented_bike_count,8760.0,704.6,645.0,0.0,191.0,504.5,1065.2,3556.0
temp_c,8760.0,12.9,11.9,-17.8,3.5,13.7,22.5,39.4
humidity_pct,8760.0,58.2,20.4,0.0,42.0,57.0,74.0,98.0
wind_speed_m_s,8760.0,1.7,1.0,0.0,0.9,1.5,2.3,7.4
visibility_m,8760.0,14368.3,6083.0,270.0,9400.0,16980.0,20000.0,20000.0
dew_point_c,8760.0,4.1,13.1,-30.6,-4.7,5.1,14.8,27.2
solar_radiation_mj_m2,8760.0,0.6,0.9,0.0,0.0,0.0,0.9,3.5
rainfall_mm,8760.0,0.1,1.1,0.0,0.0,0.0,0.0,35.0
snowfall_cm,8760.0,0.1,0.4,0.0,0.0,0.0,0.0,8.8


Now read it with domain knowledge: temperatures between about −18 and 39 °C fit Seoul. Humidity has a
minimum of **0 %**, which never happens outdoors. Visibility tops out at exactly 20,000 m in many rows (the
sensor's maximum). And `rented_bike_count` has a minimum of **0**: suspicious? We look at both below.

## 4. value_counts on season, holiday, functioning day

`for col in [...]:` repeats the indented line once for each of the three column names in the list; in each
round the variable `col` holds one name.
`df[col]` picks that **one column** from the table. `.value_counts()` counts **how often each value occurs**,
most frequent first. `print(..., "\n")` shows the result and adds an empty line (`"\n"` = new line) between
the three tables.

In [8]:
for col in ["season", "holiday", "functioning_day"]:
    print(df[col].value_counts(), "\n")

season
Spring    2208
Summer    2208
Autumn    2184
Winter    2160
Name: count, dtype: int64 

holiday
No Holiday    8328
Holiday        432
Name: count, dtype: int64 

functioning_day
Yes    8465
No      295
Name: count, dtype: int64 



Check: 4 seasons, 2 holiday values, 2 functioning-day values, and no odd spellings. Good.

## 5. Parse timestamp and derive hour, weekday, month; meet split

In a CSV file the time is just **text** like `"2017-12-01 05:00"`. `pd.to_datetime(...)` turns that text into
a real **date-time** that pandas can calculate with. We keep it in a separate variable `ts`.

In [9]:
ts = pd.to_datetime(df["timestamp"])

`.dt` opens the date-time tools of `ts`. `.dt.hour` takes the **hour** (0–23) out of every timestamp.
`df["hour"] = ...` stores the result as a **new column** called `hour`.

In [10]:
df["hour"] = ts.dt.hour

`.dt.day_of_week` gives the **weekday as a number**: 0 = Monday, 1 = Tuesday, … 6 = Sunday.
Everything after `#` is a **comment**: Python ignores it; it's a note for humans.

In [11]:
df["weekday"] = ts.dt.day_of_week   # 0 = Monday

`.dt.month` gives the **month** (1 = January … 12 = December).

In [12]:
df["month"] = ts.dt.month

`df[[...]]` with **double** brackets selects **several** columns at once (a list of names inside the
brackets). `.head()` shows the first 5 rows: we check that the new columns match the timestamp.

In [13]:
df[["timestamp", "hour", "weekday", "month"]].head()

,timestamp,hour,weekday,month
0,2017-12-01 00:00,0,4,12
1,2017-12-01 01:00,1,4,12
2,2017-12-01 02:00,2,4,12
3,2017-12-01 03:00,3,4,12
4,2017-12-01 04:00,4,4,12


`ts.min()` and `ts.max()` give the **earliest and latest** time in the data. Writing both separated by a
comma shows them together as a pair.

In [14]:
ts.min(), ts.max()

(Timestamp('2017-12-01 00:00:00'), Timestamp('2018-11-30 23:00:00'))

`split` tells us which rows belong to the **training** set and which to the **test** set (used from Session 7
on). `value_counts()` again counts the rows per value.

In [15]:
df["split"].value_counts()

split
train    6576
test     2184
Name: count, dtype: int64

`df.groupby("split")` splits the table into groups, one per value of `split`. `["timestamp"]` keeps only the
timestamp column, and `.agg(["min", "max"])` calculates the first and last timestamp **per group**.
Result: which period is train and which is test.

In [16]:
df.groupby("split")["timestamp"].agg(["min", "max"])

,min,max
split,,
test,2018-09-01 00:00,2018-11-30 23:00
train,2017-12-01 00:00,2018-08-31 23:00


**train** = Dec 2017–Aug 2018, **test** = Sep–Nov 2018: the model will later be tested on months it has
never seen.

## 6. Build the data dictionary as a small DataFrame

A **data dictionary** describes every column: what it means, which values we expect, and whether the data
matches. Meaning and expected range come from **us** (the column names and what we know about Seoul);
observed range and missing share come from the data.

First we write down our knowledge as a Python **dictionary** (`{key: value}`): for each column name a tuple
`(meaning, lowest plausible value, highest plausible value)`. `None` means "no numeric range" (text columns).

In [17]:
meaning = {
    "timestamp": ("start of the hour (local time)", None, None),
    "rented_bike_count": ("bikes rented in that hour (target)", 0, 5000),
    "temp_c": ("air temperature in °C", -25, 40),
    "humidity_pct": ("relative humidity in %", 5, 100),
    "wind_speed_m_s": ("wind speed in m/s", 0, 20),
    "visibility_m": ("visibility in m", 0, 20000),
    "dew_point_c": ("dew point temperature in °C", -35, 30),
    "solar_radiation_mj_m2": ("solar radiation in MJ/m² per hour", 0, 4),
    "rainfall_mm": ("rain in mm per hour", 0, 50),
    "snowfall_cm": ("new snow in cm", 0, 10),
    "season": ("season of the year", None, None),
    "holiday": ("public holiday or not", None, None),
    "functioning_day": ("was the rental system operating?", None, None),
    "split": ("prescribed train/test split", None, None),
}

Now we build one row of the dictionary per column. Line by line:

| Code | What it does |
|---|---|
| `rows = []` | an empty list; we will add one entry per column |
| `for col, (text, lo, hi) in meaning.items():` | go through the dictionary; each round gives the column name `col` and unpacks its tuple into `text`, `lo`, `hi` |
| `s = df[col]` | the column itself, under a short name |
| `numeric = pd.api.types.is_numeric_dtype(s)` | `True` if the column holds numbers, `False` for text |
| `observed = f"{s.min():g} to {s.max():g}" if numeric else f"{s.nunique()} values"` | for numbers: the smallest and largest value as text (`f"..."` puts values into a string, `:g` drops useless zeros); for text: how many different values (`nunique`) |
| `expected = f"{lo} to {hi}" if lo is not None else "-"` | our expected range as text, or `-` for text columns |
| `plausible = ("yes" if s.between(lo, hi).all() else "no") if lo is not None else "check values"` | `s.between(lo, hi)` checks every value; `.all()` is `True` only if **all** values are inside the range |
| `rows.append({...})` | adds one entry (a dictionary with the column's facts) to the list |
| `round(s.isna().mean() * 100, 1)` | `isna()` marks missing values as `True`; the mean of True/False is the **share** missing; × 100 = percent |
| `data_dictionary = pd.DataFrame(rows)` | turns the list of entries into a table |
| `data_dictionary` | the last line of a cell is shown as output |

In [18]:
rows = []
for col, (text, lo, hi) in meaning.items():
    s = df[col]
    numeric = pd.api.types.is_numeric_dtype(s)
    observed = f"{s.min():g} to {s.max():g}" if numeric else f"{s.nunique()} values"
    expected = f"{lo} to {hi}" if lo is not None else "-"
    plausible = ("yes" if s.between(lo, hi).all() else "no") if lo is not None else "check values"
    rows.append({"column": col, "type": str(s.dtype), "meaning": text, "expected": expected,
                 "observed": observed, "missing %": round(s.isna().mean() * 100, 1),
                 "plausible?": plausible})
data_dictionary = pd.DataFrame(rows)
data_dictionary

,column,type,meaning,expected,observed,missing %,plausible?
0,timestamp,str,start of the hour (local time),-,8760 values,0.0,check values
1,rented_bike_count,int64,bikes rented in that hour (target),0 to 5000,0 to 3556,0.0,yes
2,temp_c,float64,air temperature in °C,-25 to 40,-17.8 to 39.4,0.0,yes
3,humidity_pct,int64,relative humidity in %,5 to 100,0 to 98,0.0,no
4,wind_speed_m_s,float64,wind speed in m/s,0 to 20,0 to 7.4,0.0,yes
5,visibility_m,int64,visibility in m,0 to 20000,270 to 20000,0.0,yes
6,dew_point_c,float64,dew point temperature in °C,-35 to 30,-30.6 to 27.2,0.0,yes
7,solar_radiation_mj_m2,float64,solar radiation in MJ/m² per hour,0 to 4,0 to 3.52,0.0,yes
8,rainfall_mm,float64,rain in mm per hour,0 to 50,0 to 35,0.0,yes
9,snowfall_cm,float64,new snow in cm,0 to 10,0 to 8.8,0.0,yes


One "no": humidity. Which rows?

`df["humidity_pct"] < 5` gives `True`/`False` for every row (a **mask**). `df.loc[mask, [columns]]` keeps
only the rows where the mask is `True`, and only the listed columns.

In [19]:
df.loc[df["humidity_pct"] < 5, ["timestamp", "humidity_pct", "temp_c", "rainfall_mm"]]

,timestamp,humidity_pct,temp_c,rainfall_mm
4063,2018-05-19 07:00,0,11.4,0.0
4106,2018-05-21 02:00,0,13.9,0.0
4107,2018-05-21 03:00,0,13.0,0.0
4108,2018-05-21 04:00,0,12.4,0.0
4109,2018-05-21 05:00,0,11.9,0.0
4110,2018-05-21 06:00,0,11.4,0.0
4111,2018-05-21 07:00,0,12.3,0.0
4131,2018-05-22 03:00,0,15.5,0.0
4132,2018-05-22 04:00,0,15.6,0.0
4133,2018-05-22 05:00,0,15.7,0.0


17 hours on five days in May 2018 show exactly 0 %: almost certainly a sensor problem, not weather. A real
data issue even in the "clean" file: note it for Session 5.

## Ask the room: some hours have 0 rentals. Error or real?

`df["rented_bike_count"] == 0` is again a mask: `True` for hours with 0 rentals. We store it as `zero`.

In [20]:
zero = df["rented_bike_count"] == 0

`zero.sum()` counts the `True` values (Python counts `True` as 1 and `False` as 0), i.e. the number of
0-rental hours.

In [21]:
print("hours with 0 rentals:", zero.sum())

hours with 0 rentals: 295


`pd.crosstab(a, b)` counts how often each combination of two columns occurs. Here: 0 rentals yes/no
against `functioning_day` No/Yes. `rownames=[...]` only gives the row label a readable name.

In [22]:
pd.crosstab(zero, df["functioning_day"], rownames=["0 rentals"])

functioning_day,No,Yes
0 rentals,,
False,0,8465
True,295,0


All 0-rental hours are hours with `functioning_day = No`: the system **was not operating**. Real, not an
error, but those hours tell a model nothing about demand. What to do with them is a decision for Session 5
(and it belongs in the cleaning log).